# NewsRec Cloud GPU Worker & Batch Precompute (Google Colab T4 GPU)

This notebook provides **two modes**:
1. **Live Remote GPU Worker**: Serves FastAPI endpoints (`/embed`, `/rerank`) via Cloudflare Tunnel for real-time inference.
2. **Resumable Batch Precompute**: Offloads full dataset embedding to T4 GPU, checkpointing directly to Google Drive.

### Strict Parity & Reproducibility Guarantees:
- **Strict Version Verification**: fails on startup if `sentence-transformers`, `transformers` or `numpy` are not the pinned versions.
- **Pinned Hugging Face Model Revisions**: Embedder (`a5beb1e3e68b9ab74eb54cfd186867f64f240e1a`) and Reranker (`2cfc18c9415c912f9d8155881c133215df768a70`).
- **Token caps asserted**: `/health` reports `embedding_max_seq_length` (256) and `reranker_max_length` (512) so drift from `config.py` is visible.
- **Deterministic Seeds**: Fixed seeds (`42`) across PyTorch, NumPy, and Python.
- **fp32 Precision**: Strict IEEE 754 single precision (`np.float32`) for all stored embeddings and scores.
- **Secrets Security**: Loads `HF_TOKEN` and optional `WORKER_TOKEN` from Colab Secrets. Set `WORKER_TOKEN` or the public tunnel URL is an **open, unauthenticated GPU**.

### Known constraints
- Mode B needs a **pre-cleaned** CSV (snake_case `id` / `title` / `description` / `text_embed`), not a raw `articles_batch_*.csv`.
- The bundle's `config_hash` must equal the local `config.get_pipeline_version()`; the notebook refuses to start rather than guess.
- Mode B exports **embeddings only**. The local importer additionally expects `recommendations`, so an embeddings-only bundle covers the embedding path (which is what the local pipeline recomputes) and not the recommendation-import branch.


In [ ]:
# Step 1: Install exact pinned packages & Cloudflare Tunnel binary
# NOTE: no torch pin -- Colab's preinstalled torch is the only one matched to its CUDA
# driver. Forcing a torch version here breaks the GPU. The pins below are the ones the
# scorer actually depends on (verified by the assertions in Step 2).
!pip install -q fastapi uvicorn "sentence-transformers==6.1.0" "transformers==5.19.0"
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("Dependencies & Cloudflared installed.")
print("IMPORTANT: Go to Runtime > Restart session, then run from Step 2 onwards.")


In [ ]:
# Step 2: Strict Startup Version & Environment Verification
import sys
import os
import torch
import transformers
import sentence_transformers
import numpy as np

# Fix all seeds for deterministic outputs
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

# Colab secrets: HF_TOKEN (model download). WORKER_TOKEN is optional but STRONGLY
# recommended -- without it the public tunnel URL is an open, unauthenticated GPU.
# Generate one and put the SAME value in the local .env as REMOTE_MODEL_TOKEN.
try:
    from google.colab import userdata
    for _key in ("HF_TOKEN", "WORKER_TOKEN"):
        try:
            _val = userdata.get(_key)
        except Exception:
            _val = None
        if _val:
            os.environ[_key] = _val
            print(f"Loaded {_key} from Colab Secrets.")
except Exception:
    pass

if not os.environ.get("WORKER_TOKEN"):
    print("WARNING: WORKER_TOKEN not set -- the tunnel has NO authentication.")

# Assert pinned version requirements
REQUIRED = {
    'sentence_transformers': '6.1.0',
    'transformers': '5.19.0',
}

errors = []
for pkg, expected in REQUIRED.items():
    actual = getattr(sys.modules[pkg], '__version__', None)
    if actual != expected:
        errors.append(f"{pkg}: expected {expected}, found {actual}")
np_ver = getattr(sys.modules.get('numpy'), '__version__', '')
if not np_ver.startswith('2.'):
    errors.append(f'numpy: expected 2.x, found {np_ver}')

if errors:
    raise RuntimeError("STARTUP VERSION MISMATCH DETECTED:\n" + "\n".join(errors))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"All environment checks passed! Active Device: {device} ({gpu_name})")


In [ ]:
# Step 3: Load Neural Models with Pinned HF Revisions and Token Caps
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
EMBEDDING_REVISION = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"
RERANKER_MODEL = "BAAI/bge-reranker-base"
RERANKER_REVISION = "2cfc18c9415c912f9d8155881c133215df768a70"
EMBED_MAX_SEQ_LEN = 256   # must equal config.EMBED_MAX_SEQ_LEN
RERANK_MAX_LEN = 512      # must equal config.RERANK_MAX_LEN

print(f"Loading Embedder ({EMBEDDING_MODEL} @ {EMBEDDING_REVISION[:8]})...")
embedder = SentenceTransformer(EMBEDDING_MODEL, revision=EMBEDDING_REVISION, device=device)
embedder.max_seq_length = EMBED_MAX_SEQ_LEN  # 254 content tokens + 2 special tokens

print(f"Loading Reranker ({RERANKER_MODEL} @ {RERANKER_REVISION[:8]})...")
reranker = CrossEncoder(RERANKER_MODEL, revision=RERANKER_REVISION, max_length=RERANK_MAX_LEN, device=device)
print("Both models loaded into GPU memory with pinned revisions and fp32 precision.")


In [ ]:
# Step 4: Mode A - Launch FastAPI Worker & Cloudflare Tunnel (Live Mode)
import os
import re
import time
import threading
import subprocess
from typing import List
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI(title="NewsRec GPU Worker")

# Wall-clock time of the last real /embed or /rerank call. The keep-alive cell reads
# this so it can shut the GPU down when nobody is using it.
_last_inference_at = time.time()

# Optional shared secret. If WORKER_TOKEN is set, every /embed and /rerank call must
# carry a matching X-Worker-Token header. /health stays open so the connection test works.
WORKER_TOKEN = os.environ.get("WORKER_TOKEN", "")

def _check_token(x_worker_token: str = Header(default="")):
    if WORKER_TOKEN and x_worker_token != WORKER_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid or missing X-Worker-Token")

def _touch():
    global _last_inference_at
    _last_inference_at = time.time()

class EmbedRequest(BaseModel):
    texts: List[str]

class RerankRequest(BaseModel):
    pairs: List[List[str]]  # [[query, doc], ...]

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "embedding_model": EMBEDDING_MODEL,
        "embedding_revision": EMBEDDING_REVISION,
        "embedding_max_seq_length": int(embedder.max_seq_length),
        "reranker_model": RERANKER_MODEL,
        "reranker_revision": RERANKER_REVISION,
        "reranker_max_length": int(reranker.max_length),
        "auth_required": bool(WORKER_TOKEN),
        "last_inference_at": _last_inference_at,
    }

@app.post("/embed")
def embed(req: EmbedRequest, x_worker_token: str = Header(default="")):
    _check_token(x_worker_token)
    _touch()
    if not req.texts:
        return {"vectors": []}
    vecs = embedder.encode(req.texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    # Explicit fp32 serialization
    return {"vectors": np.asarray(vecs, dtype=np.float32).tolist()}

@app.post("/rerank")
def rerank(req: RerankRequest, x_worker_token: str = Header(default="")):
    _check_token(x_worker_token)
    _touch()
    if not req.pairs:
        return {"logits": []}
    pair_tuples = [(p[0], p[1]) for p in req.pairs]
    scores = reranker.predict(pair_tuples, batch_size=32, show_progress_bar=False)
    return {"logits": np.asarray(scores, dtype=np.float32).tolist()}

# Start uvicorn in background thread
def start_server():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

# Start Cloudflare Tunnel. Read stderr on a background thread so a slow/quiet log
# cannot block the cell forever (readline() in the main loop used to hang here).
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

print("Connecting Cloudflare Tunnel...")
public_url, _err_lines = None, []

def _drain_stderr(proc, sink):
    for line in proc.stderr:
        sink.append(line)

_reader = threading.Thread(target=_drain_stderr, args=(tunnel_proc, _err_lines), daemon=True)
_reader.start()

deadline = time.time() + 30
while time.time() < deadline and public_url is None:
    for line in list(_err_lines):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 65)
    print("YOUR CLOUD GPU WORKER IS ONLINE!")
    print(f"REMOTE URL: {public_url}")
    print("=" * 65)
    print("\nCopy these lines into your local .env file:\n")
    print(f"REMOTE_MODEL_URL={public_url}")
    if WORKER_TOKEN:
        print("REMOTE_MODEL_TOKEN=<same WORKER_TOKEN you set in Colab Secrets>\n")
    else:
        print("REMOTE_MODEL_TOKEN=            # optional, but see the warning in Step 2\n")
else:
    print("Could not start tunnel automatically. Re-run this cell.")
    print("Last cloudflared output:", "".join(_err_lines[-5:]) or "(empty)")


In [ ]:
# ============================================================
# Step 4b: KEEP-ALIVE LOOP -- Run this after Step 4
# ============================================================
# This cell runs FOREVER (until you manually interrupt it or
# disband the Colab runtime yourself).
#
# What it does every 4 minutes:
#   1. Pings the local /health endpoint to simulate activity
#   2. Runs a tiny GPU tensor op to register GPU activity
#   3. Prints a heartbeat so you can see it is alive
#
# HOW TO STOP: Click the [stop] button on this cell, or
#              Runtime > Disconnect and delete runtime.
# ============================================================

import urllib.request
import json as _json
import time
import datetime

HEARTBEAT_INTERVAL_SECONDS = 240  # 4 min -- well within Colab 90-min idle timeout
WORKER_LOCAL_URL = "http://127.0.0.1:8000"

# The GPU should not stay awake for nobody. If no /embed or /rerank call has arrived
# for this long, the loop exits and loses the runtime rather than burning GPU hours.
IDLE_EXIT_SECONDS = 3600          # 1 hour of zero inference traffic -> stop

print("Keep-alive loop started. This cell runs until YOU stop it.")
print(f"   Heartbeat every {HEARTBEAT_INTERVAL_SECONDS // 60} minutes.")
print(f"   Auto-exits after {IDLE_EXIT_SECONDS // 60} minutes with no inference traffic.")
print("   Stop this cell to shut down. Runtime stays alive as long as this runs.\n")

heartbeat_count = 0
start_time = time.time()

while True:
    heartbeat_count += 1
    now = datetime.datetime.now().strftime("%H:%M:%S")
    elapsed_min = int((time.time() - start_time) / 60)

    # 1. Ping local FastAPI health endpoint
    health_status = "UNKNOWN"
    try:
        req = urllib.request.Request(
            f"{WORKER_LOCAL_URL}/health",
            headers={"User-Agent": "NewsRec-Keepalive"}
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = _json.loads(resp.read().decode("utf-8"))
            health_status = "OK" if data.get("status") == "ok" else "DEGRADED"
            last_inference = data.get("last_inference_at")
    except Exception as e:
        health_status = f"ERROR ({e})"
        last_inference = None

    # 2. Idle watchdog -- exit instead of holding the GPU idle indefinitely
    if last_inference:
        idle_for = time.time() - last_inference
        if idle_for > IDLE_EXIT_SECONDS:
            print(f"[--] [{now}] Idle for {int(idle_for)}s with no inference traffic. Stopping keep-alive.")
            break
    else:
        idle_for = 0

    # 3. Tiny GPU pulse -- keeps CUDA context warm and signals GPU usage to Colab
    gpu_info = ""
    try:
        if torch.cuda.is_available():
            _pulse = torch.zeros(64, 64, device="cuda") + 1.0
            _ = float(_pulse.sum().item())
            mem_mb = torch.cuda.memory_allocated() / 1024 / 1024
            gpu_info = f" | GPU mem: {mem_mb:.0f} MB"
            del _pulse
    except Exception:
        pass

    # 4. Heartbeat status line
    status_icon = "[OK]" if health_status == "OK" else "[!!]"
    print(
        f"{status_icon} [{now}] Heartbeat #{heartbeat_count} | "
        f"Worker: {health_status} | "
        f"Uptime: {elapsed_min}m{gpu_info}"
    )

    time.sleep(HEARTBEAT_INTERVAL_SECONDS)


In [ ]:
# ============================================================
# Step 5: CONNECTION TEST -- Run this ANYTIME to verify the
#         tunnel is properly connected to your local machine
# ============================================================
# Tests the FULL end-to-end chain:
#   Public Tunnel URL -> Cloudflare -> Colab FastAPI -> Models
#
# Paste the REMOTE_MODEL_URL from your .env file below if
# public_url is not set (i.e. you are running this cell alone).
# ============================================================

import urllib.request
import json as _json
import numpy as np
import time
import os

# Auto-reads from Step 4 if run in order; otherwise paste your URL
TEST_URL = public_url if 'public_url' in dir() and public_url else "PASTE_YOUR_URL_HERE"
TEST_TOKEN = os.environ.get("WORKER_TOKEN", "")

def _headers(extra=None):
    h = {"User-Agent": "NewsRec-ConnectionTest"}
    if TEST_TOKEN:
        h["X-Worker-Token"] = TEST_TOKEN
    if extra:
        h.update(extra)
    return h

def _post(endpoint, payload_dict, timeout=30):
    payload = _json.dumps(payload_dict).encode("utf-8")
    req = urllib.request.Request(
        f"{TEST_URL.rstrip('/')}{endpoint}",
        data=payload,
        headers=_headers({"Content-Type": "application/json"})
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return _json.loads(resp.read().decode("utf-8"))

def _get(endpoint, timeout=10):
    req = urllib.request.Request(f"{TEST_URL.rstrip('/')}{endpoint}", headers=_headers())
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return _json.loads(resp.read().decode("utf-8"))

print("=" * 60)
print(f"CONNECTION TEST against: {TEST_URL}")
print("=" * 60)

all_passed = True

# -- Test 1: /health ---------------------------------------------------
print("\n[TEST 1/3] /health endpoint...")
try:
    t0 = time.time()
    health = _get("/health")
    latency = (time.time() - t0) * 1000
    assert health.get("status") == "ok", f"Expected status=ok, got: {health}"
    print(f"  PASS -- status=ok | device={health.get('device')} | gpu={health.get('gpu')} | latency={latency:.0f}ms")
    print(f"     embed_model : {health.get('embedding_model')}")
    print(f"     rerank_model: {health.get('reranker_model')}")
    EXPECTED_EMBED_REV  = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"
    EXPECTED_RERANK_REV = "2cfc18c9415c912f9d8155881c133215df768a70"
    rev_embed = health.get("embedding_revision", "")
    rev_rerank = health.get("reranker_revision", "")
    ok_e = rev_embed == EXPECTED_EMBED_REV
    ok_r = rev_rerank == EXPECTED_RERANK_REV
    print("     embed_revision  :", "MATCH" if ok_e else f"MISMATCH got {rev_embed}")
    print("     rerank_revision :", "MATCH" if ok_r else f"MISMATCH got {rev_rerank}")
    if not (ok_e and ok_r):
        all_passed = False
    # Token-cap drift is a silent quality regression: the local scorer assumes
    # 256 / 512, so a runtime with different caps produces incomparable scores.
    seq_len = health.get("embedding_max_seq_length")
    rk_len  = health.get("reranker_max_length")
    print(f"     embed_max_seq_len: {seq_len} (expect 256) | reranker_max_len: {rk_len} (expect 512)")
    if seq_len != 256 or rk_len != 512:
        print("     WARNING: token caps differ from config -- scores will not match local.")
        all_passed = False
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Test 2: /embed ----------------------------------------------------
print("\n[TEST 2/3] /embed endpoint (2 test sentences)...")
TEST_SENTENCES = [
    "Virat Kohli hits his 50th ODI century at Wankhede Stadium",
    "Supreme Court hears plea on electoral bond transparency"
]
try:
    t0 = time.time()
    result = _post("/embed", {"texts": TEST_SENTENCES})
    latency = (time.time() - t0) * 1000
    vectors = np.array(result["vectors"], dtype=np.float32)
    assert vectors.shape == (2, 768), f"Expected shape (2, 768), got {vectors.shape}"
    norms = np.linalg.norm(vectors, axis=1)
    assert all(abs(n - 1.0) < 0.01 for n in norms), f"Vectors not normalized! norms={norms}"
    cosine_sim = float(vectors[0] @ vectors[1])
    print(f"  PASS -- shape={vectors.shape} | dtype=float32 | latency={latency:.0f}ms")
    print(f"     Norms: {norms[0]:.4f}, {norms[1]:.4f}  (should be ~1.0)")
    print(f"     Cosine similarity Sports vs Politics: {cosine_sim:.4f}  (expect ~0.2-0.5)")
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Test 3: /rerank ---------------------------------------------------
print("\n[TEST 3/3] /rerank endpoint (relevant pair + irrelevant pair)...")
RELEVANT_PAIR   = ["Kohli ODI century record Wankhede", "Virat Kohli scores 50th ODI hundred, breaks all-time record"]
IRRELEVANT_PAIR = ["Kohli ODI century record Wankhede", "Budget 2025: Finance ministry plans new tax slab revision"]
try:
    t0 = time.time()
    result = _post("/rerank", {"pairs": [RELEVANT_PAIR, IRRELEVANT_PAIR]})
    latency = (time.time() - t0) * 1000
    logits = np.array(result["logits"], dtype=np.float32)
    assert logits.shape == (2,), f"Expected shape (2,), got {logits.shape}"
    probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    print(f"  PASS -- shape={logits.shape} | latency={latency:.0f}ms")
    print(f"     Relevant pair   -- logit: {logits[0]:.4f} | prob: {probs[0]:.4f}")
    print(f"     Irrelevant pair -- logit: {logits[1]:.4f} | prob: {probs[1]:.4f}")
    sanity = "PASS" if probs[0] > probs[1] else "FAIL -- relevant scored lower than irrelevant"
    print(f"     Sanity check: {sanity}")
    if probs[0] <= probs[1]:
        all_passed = False
except Exception as e:
    print(f"  FAIL -- {e}")
    all_passed = False

# -- Final verdict -----------------------------------------------------
print("\n" + "=" * 60)
if all_passed:
    print("ALL TESTS PASSED -- Colab worker is fully connected!")
    print("   Set in your local .env:")
    print(f"   REMOTE_MODEL_URL={TEST_URL}")
    if TEST_TOKEN:
        print("   REMOTE_MODEL_TOKEN=<same token>")
else:
    print("SOME TESTS FAILED -- check errors above.")
    print("   Common causes:")
    print("   1. Tunnel not started  -- run Step 4 first")
    print("   2. Wrong URL           -- copy REMOTE URL printed by Step 4")
    print("   3. Runtime disconnected -- restart and re-run all steps")
print("=" * 60)


In [ ]:
# Step 6: Mode B - Resumable Batch Precompute with Google Drive Checkpoints
# Precompute embeddings offline on the T4 and export a bundle that
# import_colab_export.py will accept.
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/newsrec_colab")
CHECKPOINT_EVERY = 50

# Point this at the PRE-CLEANED corpus CSV (snake_case columns: id, title,
# description, text_embed). Generate it locally with src.data_loader.clean_dataframe();
# a raw articles_batch_*.csv has camelCase columns and will raise KeyError here.
BATCH_CSV_PATH = "/content/drive/MyDrive/newsrec_colab/articles_clean.csv"

def mount_drive_if_needed():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Checkpoints will be saved to: {CHECKPOINT_DIR}")
    except Exception as e:
        print(f"Notice: Drive mount skipped ({e}). Using local session storage.")

def run_batch_precompute(csv_path: str = BATCH_CSV_PATH):
    mount_drive_if_needed()
    base_dir = CHECKPOINT_DIR if CHECKPOINT_DIR.exists() else Path("newsrec_colab")
    base_dir.mkdir(parents=True, exist_ok=True)
    target_file = base_dir / "checkpoint.json"
    export_file = base_dir / "newsrec_export.json"

    # The hash must match config.get_pipeline_version() locally, or
    # import_colab_export.py rejects the bundle. Compute it locally, then set it here:
    #   python -c "import config; print(config.get_pipeline_version())"
    pinned = os.environ.get("NEWSREC_PIPELINE_VERSION")
    if not pinned:
        raise RuntimeError(
            "NEWSREC_PIPELINE_VERSION is not set. Run locally "
            '"python -c \"import config; print(config.get_pipeline_version())\"", '
            "then set os.environ['NEWSREC_PIPELINE_VERSION'] = '<hash>' in this cell."
        )

    data = {"config_hash": pinned, "articles": {}}
    if target_file.exists():
        with open(target_file, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if saved.get("config_hash") == pinned:
            data = saved
            print(f"Resuming from checkpoint: {len(data['articles'])} articles already processed.")
        else:
            # Never silently throw away completed work -- this used to discard the
            # previous checkpoint and overwrite it.
            raise RuntimeError(
                f"Checkpoint holds config_hash={saved.get('config_hash')} but "
                f"NEWSREC_PIPELINE_VERSION={pinned}. Refusing to discard "
                f"{len(saved.get('articles', {}))} finished articles. "
                f"Delete or move {target_file} to start over."
            )

    df = pd.read_csv(csv_path)
    missing = {"id", "title"} - set(df.columns)
    if missing:
        raise KeyError(
            f"CSV is missing columns {sorted(missing)}. Expected the snake_case output of "
            f"src.data_loader.clean_dataframe(); got {sorted(df.columns)[:12]}"
        )
    print(f"Total articles in CSV: {len(df)}")

    todo = [row for _, row in df.iterrows() if str(row["id"]) not in data["articles"]]
    print(f"Articles to process: {len(todo)}")

    # Batched encode (32 at a time) instead of one call per article.
    ids    = [str(row["id"]) for row in todo]
    titles = [str(row["title"]) for row in todo]
    texts  = [
        str(row.get("text_embed") or f"{row['title']}. {row.get('description', '')}")
        for row in todo
    ]

    vectors = np.asarray(
        embedder.encode(
            texts, batch_size=32, normalize_embeddings=True,
            show_progress_bar=True, convert_to_numpy=True,
        ) if texts else [],
        dtype=np.float32,
    )

    for n, (aid, title) in enumerate(zip(ids, titles), start=1):
        data["articles"][aid] = {"embedding": vectors[n - 1].tolist(), "title": title}
        if n % CHECKPOINT_EVERY == 0:
            with open(target_file, "w", encoding="utf-8") as f:
                json.dump(data, f)
            print(f"Checkpointed {len(data['articles'])}/{len(df)} articles to Drive.")

    with open(export_file, "w", encoding="utf-8") as f:
        json.dump(data, f)
    print(f"Export complete! {len(data['articles'])} articles -> {export_file}")
    print(f"config_hash = {data['config_hash']}")

print("Batch precompute ready. Set NEWSREC_PIPELINE_VERSION, then call run_batch_precompute()")
